# Name That LLM: offline submission

Attach a Kaggle Dataset built with kaggle/prepare_artifact.py, or attach its verified kaggle_artifact.zip. ZIP inputs are extracted into /kaggle/working before inference. In notebook settings, select CPU (accelerator None) for the provided CPU-only PyTorch wheelhouse and turn Internet off. This notebook runs inference only; it does not download weights or training data.

The competition rerun replaces the small interactive test file with about 4,000 hidden rows. We therefore use the official test.csv and do not compare against the interactive sample_submission.parquet. The final script checks IDs, labels, and a Parquet round trip.

Official format and runtime rules: https://www.kaggle.com/competitions/name-that-llm/overview/abstract

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
COMPETITION_DIR = INPUT_ROOT / "competitions" / "name-that-llm"
ARTIFACT_ROOT = None  # Set to the mounted Kaggle Dataset directory if autodetection is ambiguous.
ARTIFACT_ZIP = None  # Optional path to an attached ZIP if multiple model ZIPs are mounted.
WHEELHOUSE_DIR = None  # Auto-filled from an attached extracted wheelhouse directory or ZIP.
WHEELHOUSE_ZIP = None  # Optional explicit ZIP path if autodetection is ambiguous.
OUTPUT_PATH = Path("/kaggle/working/submission.parquet")

In [ ]:
import hashlib
import json
import zipfile

RECIPE = "e0c_plus_roberta_12ep_scalerfix_equal_rank"
SOURCE_FILES = ("final_pipeline.py", "run_e0.py", "run_style.py",
                "run_combined.py", "run_fixed_blend.py")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

ARTIFACT_ARCHIVE = None
if ARTIFACT_ROOT is None:
    candidates = []
    for item in INPUT_ROOT.rglob("artifact_manifest.json"):
        try:
            candidate = json.loads(item.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if candidate.get("recipe") == RECIPE:
            candidates.append(item.parent)
    if not candidates:
        zip_candidates = []
        for item in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(item) as bundle:
                    if "artifact_manifest.json" in bundle.namelist():
                        zip_candidates.append(item)
            except (OSError, zipfile.BadZipFile):
                continue
        if ARTIFACT_ZIP is not None:
            zip_candidates = [Path(ARTIFACT_ZIP)]
        if len(zip_candidates) != 1:
            raise RuntimeError(
                f"Expected one model artifact ZIP, found {len(zip_candidates)}: {zip_candidates}. "
                "Set ARTIFACT_ZIP in the config cell."
            )
        ARTIFACT_ARCHIVE = Path(zip_candidates[0]).resolve(strict=True)
        extracted_root = Path("/kaggle/working/name_that_llm_artifact")
        with zipfile.ZipFile(zip_candidates[0]) as bundle:
            names = bundle.namelist()
            if len(names) != len(set(names)) or "artifact_manifest.json" not in names:
                raise RuntimeError("Invalid model artifact ZIP")
            if any(name.startswith("/") or ".." in name.split("/") or "\\" in name for name in names):
                raise RuntimeError("Unsafe path in model artifact ZIP")
            bundle.extractall(extracted_root)
        candidates = [extracted_root]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected one model artifact Dataset, found {len(candidates)}: {candidates}. "
            "Set ARTIFACT_ROOT in the config cell."
        )
    ARTIFACT_ROOT = candidates[0]
ARTIFACT_ROOT = Path(ARTIFACT_ROOT).resolve(strict=True)
manifest = json.loads((ARTIFACT_ROOT / "artifact_manifest.json").read_text(encoding="utf-8"))
if manifest.get("format_version") != 1 or manifest.get("recipe") != RECIPE:
    raise RuntimeError("Unexpected artifact manifest")
for relative_name, expected in manifest["sha256"].items():
    path = (ARTIFACT_ROOT / relative_name).resolve(strict=True)
    if not path.is_relative_to(ARTIFACT_ROOT) or sha256_file(path) != expected:
        raise RuntimeError(f"Artifact hash mismatch: {relative_name}")

SOURCE_DIR = (ARTIFACT_ROOT / manifest["source_dir"]).resolve(strict=True)
PACKAGE_DIR = (ARTIFACT_ROOT / manifest["package_dir"]).resolve(strict=True)
if not SOURCE_DIR.is_relative_to(ARTIFACT_ROOT) or not PACKAGE_DIR.is_relative_to(ARTIFACT_ROOT):
    raise RuntimeError("Artifact source/package directory escapes the mounted Dataset")
metadata = json.loads((PACKAGE_DIR / "metadata.json").read_text(encoding="utf-8"))
if metadata.get("recipe") != RECIPE or metadata.get("mode") != "full":
    raise RuntimeError("Expected the trained full package")
if set(metadata.get("source_sha256", {})) != set(SOURCE_FILES):
    raise RuntimeError("Unexpected source manifest in model package")
for name in SOURCE_FILES:
    if sha256_file(SOURCE_DIR / name) != metadata["source_sha256"][name]:
        raise RuntimeError(f"Source differs from model training: {name}")
TEST_PATH = COMPETITION_DIR / "test.csv"
if not TEST_PATH.is_file():
    raise FileNotFoundError(f"Official competition test file missing: {TEST_PATH}")
print("Verified full model package and source hashes:", ARTIFACT_ROOT)

In [ ]:
import importlib.metadata as importlib_metadata
import os
import platform
import shutil
import subprocess
import sys

WHEELHOUSE_ARCHIVE_SHA256 = "2c4dc134de5717ed8d974c4fb8405f44d848045c08b08706826436ec66480a84"

def installed_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def runtime_issues():
    issues = []
    exact = {
        "scikit-learn": metadata["sklearn_version"],
        "torch": metadata["torch_version"].split("+")[0],
        "transformers": metadata["transformers_version"],
    }
    for name, expected in exact.items():
        actual = installed_version(name)
        comparable = actual.split("+")[0] if actual and name == "torch" else actual
        if comparable != expected:
            issues.append(f"{name}: installed={actual}, required={expected}")
    for name in ("numpy", "pandas", "joblib", "safetensors", "pyarrow"):
        if installed_version(name) is None:
            issues.append(f"{name}: missing")
    return issues

PYTHON_EXECUTABLE = Path(sys.executable)
issues = runtime_issues()
if issues:
    if WHEELHOUSE_DIR is None:
        directory_candidates = [
            item.parent for item in INPUT_ROOT.rglob("requirements-kaggle-offline.txt")
            if (item.parent / "manifest.json").is_file()
        ]
        if len(directory_candidates) > 1:
            siblings = [item for item in directory_candidates if item.parent == ARTIFACT_ROOT.parent]
            if siblings:
                directory_candidates = siblings
        if len(directory_candidates) == 1:
            WHEELHOUSE_DIR = directory_candidates[0]
            print("Found extracted wheelhouse Dataset:", WHEELHOUSE_DIR)
        elif directory_candidates:
            raise RuntimeError(
                f"Found multiple wheelhouse directories: {directory_candidates}. "
                "Set WHEELHOUSE_DIR in the config cell."
            )
        else:
            if WHEELHOUSE_ZIP is None:
                wheelhouse_candidates = list(INPUT_ROOT.rglob("kaggle_wheelhouse_cp313_linux_x86_64.zip"))
                if ARTIFACT_ARCHIVE is not None:
                    siblings = [item for item in wheelhouse_candidates if item.parent == ARTIFACT_ARCHIVE.parent]
                    if siblings:
                        wheelhouse_candidates = siblings
                if len(wheelhouse_candidates) != 1:
                    raise RuntimeError(
                        f"Dependency mismatch: {issues}. Found {len(wheelhouse_candidates)} wheelhouse ZIPs: "
                        f"{wheelhouse_candidates}. Set WHEELHOUSE_DIR or WHEELHOUSE_ZIP."
                    )
                wheelhouse_zip = wheelhouse_candidates[0]
            else:
                wheelhouse_zip = Path(WHEELHOUSE_ZIP)
            wheelhouse_zip = wheelhouse_zip.resolve(strict=True)
            if sha256_file(wheelhouse_zip) != WHEELHOUSE_ARCHIVE_SHA256:
                raise RuntimeError("Wheelhouse ZIP SHA-256 differs from the validated upload")
            extracted_wheels = Path("/kaggle/working/name_that_llm_wheelhouse")
            with zipfile.ZipFile(wheelhouse_zip) as bundle:
                names = bundle.namelist()
                if len(names) != len(set(names)) or any(
                    "/" in name or "\\" in name or name in (".", "..") for name in names
                ):
                    raise RuntimeError("Unsafe or duplicate path in wheelhouse ZIP")
                bundle.extractall(extracted_wheels)
            WHEELHOUSE_DIR = extracted_wheels
            print("Extracted offline wheelhouse:", WHEELHOUSE_DIR)
    WHEELHOUSE_DIR = Path(WHEELHOUSE_DIR).resolve(strict=True)
    if not WHEELHOUSE_DIR.is_dir():
        raise NotADirectoryError(WHEELHOUSE_DIR)
    if sys.version_info[:2] != (3, 13) or sys.platform != "linux" or platform.machine() != "x86_64":
        raise RuntimeError(
            "Attached wheelhouse requires CPython 3.13 on Linux x86_64; "
            f"found Python {sys.version.split()[0]} on {sys.platform}/{platform.machine()}"
        )
    manifest_rows = json.loads((WHEELHOUSE_DIR / "manifest.json").read_text(encoding="utf-8"))
    expected_wheels = {row["file"]: row for row in manifest_rows}
    expected_names = set(expected_wheels) | {
        "manifest.json", "README.txt", "SHA256SUMS.txt", "requirements-kaggle-offline.txt"
    }
    if (
        len(manifest_rows) != 35
        or len(expected_wheels) != 35
        or any(not name.endswith(".whl") or "/" in name or "\\" in name for name in expected_wheels)
        or {item.name for item in WHEELHOUSE_DIR.iterdir()} != expected_names
    ):
        raise RuntimeError("Wheelhouse directory layout differs from its 35-wheel manifest")
    for name, item in expected_wheels.items():
        wheel = WHEELHOUSE_DIR / name
        if wheel.stat().st_size != item["bytes"] or sha256_file(wheel) != item["sha256"]:
            raise RuntimeError(f"Wheelhouse integrity check failed: {name}")
    print("Verified all 35 offline wheel hashes")
    requirement_file = WHEELHOUSE_DIR / "requirements-kaggle-offline.txt"
    if not requirement_file.is_file() or "torch==2.6.0+cpu" not in requirement_file.read_text(encoding="utf-8"):
        raise RuntimeError("Expected the validated CPU-only Kaggle wheelhouse requirements")
    # Kaggle's system Python may fail ensurepip. Build a pip-free venv, then
    # let a host pip >=22.3 manage that interpreter (pip's documented --python option).
    venv_dir = Path("/kaggle/working/name_that_llm_venv_nodeps")
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(venv_dir)], check=True)
    venv_python = venv_dir / "bin" / "python"
    if not venv_python.is_file():
        raise RuntimeError(f"Virtual environment interpreter missing: {venv_python}")
    pip_frontend = [sys.executable, "-m", "pip"]
    pip_probe = subprocess.run(pip_frontend + ["--version"], capture_output=True, text=True)
    if pip_probe.returncode:
        pip_command = shutil.which("pip")
        if pip_command is None:
            raise RuntimeError("No system pip is available to bootstrap the pip-free venv offline")
        pip_frontend = [pip_command]
    # Install every verified wheel explicitly. The hub package declares hf-xet
    # for remote transfers, but that wheel is absent and inference is local-only.
    wheel_paths = [str(WHEELHOUSE_DIR / name) for name in sorted(expected_wheels)]
    subprocess.run(
        pip_frontend + ["--python", str(venv_dir), "install", "--no-index", "--no-deps"]
        + wheel_paths,
        check=True,
    )
    probe = (
        "import sys, numpy, pandas, sklearn, joblib, torch, transformers, pyarrow, safetensors; "
        "from transformers import AutoConfig, AutoTokenizer; "
        f"assert sklearn.__version__ == {metadata['sklearn_version']!r}; "
        f"assert torch.__version__.split('+')[0] == {metadata['torch_version'].split('+')[0]!r}; "
        f"assert transformers.__version__ == {metadata['transformers_version']!r}; "
        "AutoConfig.from_pretrained(sys.argv[1], local_files_only=True); "
        "AutoTokenizer.from_pretrained(sys.argv[2], local_files_only=True, use_fast=True); "
        "print('Offline venv and local model loading ready:', torch.__version__)"
    )
    probe_environment = os.environ.copy()
    probe_environment["HF_HUB_OFFLINE"] = "1"
    probe_environment["TRANSFORMERS_OFFLINE"] = "1"
    probe_environment["HF_HUB_DISABLE_XET"] = "1"
    subprocess.run(
        [str(venv_python), "-c", probe, str(PACKAGE_DIR / "encoder_config"),
         str(PACKAGE_DIR / "tokenizer")],
        env=probe_environment, check=True,
    )
    PYTHON_EXECUTABLE = venv_python
else:
    print("Notebook runtime already matches the fitted package; using", PYTHON_EXECUTABLE)


In [ ]:
import os
import subprocess
import sys

if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
environment = os.environ.copy()
environment["HF_HUB_OFFLINE"] = "1"
environment["HF_HUB_DISABLE_XET"] = "1"
environment["TRANSFORMERS_OFFLINE"] = "1"
environment["TOKENIZERS_PARALLELISM"] = "false"
threads = str(min(4, os.cpu_count() or 4))
environment["OMP_NUM_THREADS"] = threads
environment["MKL_NUM_THREADS"] = threads
command = [
    str(PYTHON_EXECUTABLE), str(SOURCE_DIR / "final_pipeline.py"), "submit",
    "--full-package", str(PACKAGE_DIR),
    "--test", str(TEST_PATH),
    "--output", str(OUTPUT_PATH),
]
subprocess.run(command, cwd=SOURCE_DIR, env=environment, check=True, timeout=8 * 60 * 60)
if not OUTPUT_PATH.is_file():
    raise RuntimeError("Inference completed without submission.parquet")
subprocess.run(
    [str(PYTHON_EXECUTABLE), "-c",
     "import sys, pyarrow.parquet as pq; print('Ready:', sys.argv[1], 'rows:', pq.read_metadata(sys.argv[1]).num_rows)",
     str(OUTPUT_PATH)],
    check=True,
)